# Pan-zonal HepatoNet FastCORE reduction workflow

        This notebook reproduces the pan-zonal reduced metabolic network used for the PhysiCell-dFBA hepatic lobule simulations.

        The workflow has three stages:

        1. prepare a pan-zonal functional core from the reduced carbohydrate/lactate/oxidative modules;
        2. run the official FastCORE reduction around that core;
        3. validate that the reduced SBML still supports glucose/G6P, lactate/pyruvate, and oxidative CO2-producing modules.

        The generated network is the file used as the current pan-zonal dFBA model:

        `config/hepatonet_pan_zonal_official_fastcore_reduced.xml`

## GitHub setup for Colab

        If you are running this notebook from Google Colab, run this cell first.

        Change `REPO_URL` only if the files are hosted in a different GitHub repository.

In [ ]:
# Colab only: clone the GitHub repository that contains config/ and notebooks/.
        import os
        import subprocess
        import sys
        from pathlib import Path

        REPO_URL = "https://github.com/MatheusAmorim7/archivesforfastcore.git"
        REPO_DIR = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

        if "google.colab" in sys.modules:
            if not REPO_DIR.exists():
                subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
            os.chdir(REPO_DIR)
            print("Running from:", Path.cwd())
        else:
            print("Not running in Colab; keeping current folder:", Path.cwd())

## 0. Dependencies

        In the local WSL environment, use the virtual environment created for this project.
        In Colab, uncomment the install line before running the rest of the notebook.

In [ ]:
# Colab only:
        # !pip -q install cobra python-libsbml swiglpk pandas

## 1. Locate the project

        This cell works when the notebook is inside the project, inside `notebooks/`, or in a cloned GitHub repository with the same folder structure.

In [ ]:
from pathlib import Path
        import os
        import shutil
        import subprocess
        import sys

        import pandas as pd

        cwd = Path.cwd().resolve()
        if cwd.name == "notebooks":
            ROOT = cwd.parent
        elif (cwd / "config").exists() and (cwd / "notebooks").exists():
            ROOT = cwd
        elif Path("/content/archivesforfastcore").exists():
            ROOT = Path("/content/archivesforfastcore")
        else:
            ROOT = cwd

        os.chdir(ROOT)
        sys.path.insert(0, str(ROOT / "notebooks"))

        print("Project root:", ROOT)
        print("Config folder exists:", (ROOT / "config").exists())
        print("Notebooks folder exists:", (ROOT / "notebooks").exists())

## 2. Required files

        The pan-zonal workflow uses the original HepatoNet SBML as input, then creates a functional core and the official FastCORE-reduced SBML.

        Required project files:

        - `notebooks/run_pan_zonal_reviewer_fastcore.py`
        - `notebooks/run_pan_zonal_official_fastcore.py`
        - `notebooks/run_official_fastcore_literature.py`
        - `notebooks/run_literature_core_tests.py`
        - `notebooks/fastcore_outputs/reviewer_minimal_carbohydrate_core.csv`
        - `config/hepatonet_original_13_08.xml`
        - `config/PhysiCell_settings.xml`

        If `config/hepatonet_original_13_08.xml` is missing locally but the original Windows file is present on Desktop or Downloads, this cell copies it into `config/` with the expected filename.

In [ ]:
expected_original = ROOT / "config" / "hepatonet_original_13_08.xml"
        windows_candidates = [
            Path("/mnt/c/Users/Dell/Desktop/hepatonet original 13-08.xml"),
            Path("/mnt/c/Users/Dell/Downloads/hepatonet original 13-08.xml"),
        ]

        if not expected_original.exists():
            for candidate in windows_candidates:
                if candidate.exists():
                    expected_original.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(candidate, expected_original)
                    print("Copied original HepatoNet SBML from:", candidate)
                    break

        required = [
            ROOT / "notebooks" / "run_pan_zonal_reviewer_fastcore.py",
            ROOT / "notebooks" / "run_pan_zonal_official_fastcore.py",
            ROOT / "notebooks" / "run_official_fastcore_literature.py",
            ROOT / "notebooks" / "run_literature_core_tests.py",
            ROOT / "notebooks" / "fastcore_outputs" / "reviewer_minimal_carbohydrate_core.csv",
            expected_original,
            ROOT / "config" / "PhysiCell_settings.xml",
        ]

        missing = [str(path) for path in required if not path.exists()]
        if missing:
            print("Missing files:")
            for path in missing:
                print(" -", path)
            raise FileNotFoundError("Some required files are missing. Add them before continuing.")

        print("All required files were found.")

## 3. Build the pan-zonal functional core

        This step prepares the pan-zonal HepatoNet model and creates the core reaction table used as FastCORE input.

        It tests/supports the modules requested for a reduced metabolic interpretation:

        - glucose uptake to G6P;
        - G6P to glucose release;
        - lactate to pyruvate;
        - pyruvate to lactate;
        - oxidative glucose module represented by O2 uptake and CO2 release.

In [ ]:
cmd = [sys.executable, str(ROOT / "notebooks" / "run_pan_zonal_reviewer_fastcore.py")]
        result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

        pan_out = ROOT / "notebooks" / "fastcore_outputs" / "pan_zonal_reviewer_core"
        core_csv = pan_out / "pan_zonal_reviewer_core_reactions.csv"
        validation_csv = pan_out / "pan_zonal_reviewer_validation.csv"

        print("Core CSV:", core_csv)
        print("Validation CSV:", validation_csv)

In [ ]:
core = pd.read_csv(core_csv)
        validation = pd.read_csv(validation_csv)

        print("Core reactions:", len(core))
        display(validation)
        display(core.head(20))

## 4. Run official FastCORE

        This step runs the FastCORE implementation and then preserves the complete validated functional core when FastCORE alone removes reactions that are required by the reduced module tests.

        The final reduced model should contain all input core reactions and pass all validation tests.

In [ ]:
cmd = [sys.executable, str(ROOT / "notebooks" / "run_pan_zonal_official_fastcore.py")]
        result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

        official_out = pan_out / "official_fastcore"
        summary_csv = official_out / "pan_zonal_official_fastcore_summary.csv"
        official_validation_csv = official_out / "pan_zonal_official_fastcore_validation.csv"
        reduced_xml = official_out / "hepatonet_pan_zonal_official_fastcore_reduced.xml"

        print("Reduced XML:", reduced_xml)

In [ ]:
summary = pd.read_csv(summary_csv)
        official_validation = pd.read_csv(official_validation_csv)

        display(summary)
        display(official_validation)

        assert bool(summary.loc[0, "validation_all_optimal"]) is True
        assert (official_validation["status"] == "optimal").all()
        print("Validation passed.")

## 5. Optional: prepare the SBML for PhysiCell

        This optional cell copies the validated FastCORE SBML into `config/` and defines `EX_HC00021_s` as the objective reaction for dFBA coupling.

        In the current PhysiCell-dFBA configuration, `EX_HC00021_s` is used as a practical objective because it keeps the oxidative module active and directly reports CO2 release. The actual zonal behavior is still driven by local substrate concentrations and transport bounds during simulation.

In [ ]:
COPY_TO_CONFIG = False  # change to True when you want to update config/

        if COPY_TO_CONFIG:
            import cobra

            dst = ROOT / "config" / "hepatonet_pan_zonal_official_fastcore_reduced.xml"
            model = cobra.io.read_sbml_model(str(reduced_xml))
            model.solver = "glpk"
            model.objective = "EX_HC00021_s"
            model.objective_direction = "max"
            cobra.io.write_sbml_model(model, str(dst))
            print("Copied and prepared:", dst)
            print("Objective:", model.objective.expression)
            print("Direction:", model.objective.direction)
        else:
            print("No file was copied. Set COPY_TO_CONFIG = True to update config/.")

## 6. Static PhysiCell coupling check

        This checks whether the exchange reaction IDs used in `PhysiCell_settings.xml` are present in the reduced SBML.

In [ ]:
import xml.etree.ElementTree as ET

        def sbml_reaction_ids(path):
            root = ET.parse(path).getroot()
            ids = set()
            for elem in root.iter():
                if elem.tag.endswith("reaction") and "id" in elem.attrib:
                    ids.add(elem.attrib["id"])
            return ids

        settings = ROOT / "config" / "PhysiCell_settings.xml"
        sbml_ids = sbml_reaction_ids(reduced_xml)

        rows = []
        root = ET.parse(settings).getroot()
        for ex in root.findall(".//transport_model/exchange"):
            substrate = ex.attrib.get("substrate", "")
            flux_node = ex.find("fba_flux")
            if flux_node is None or not flux_node.text:
                continue
            settings_id = flux_node.text.strip()
            cobra_id = settings_id[2:] if settings_id.startswith("R_") else settings_id
            rows.append({
                "substrate": substrate,
                "settings_fba_flux": settings_id,
                "present_in_reduced_sbml": settings_id in sbml_ids or cobra_id in sbml_ids,
            })

        coupling = pd.DataFrame(rows).drop_duplicates()
        display(coupling)
        assert coupling["present_in_reduced_sbml"].all()
        print("All PhysiCell exchange IDs were found in the reduced SBML.")

## Notes for the report

        The pan-zonal reduced network was derived from HepatoNet1 using FastCORE. The core reaction set was defined to preserve a compact functional representation of carbohydrate metabolism and lactate/oxidative modules, including glucose, G6P, glycogen-related reactions, pyruvate, lactate, O2, CO2, and the exchange reactions used by the PhysiCell-dFBA coupling.

        The reduction was performed as a single pan-zonal model rather than three separate zonal SBML files. Therefore, zonal behavior is not imposed by separate metabolic networks; it emerges during simulation from local substrate concentrations, microenvironmental gradients, and transport constraints.

        Before coupling to PhysiCell, the reduced SBML was validated by forcing normalized flux through the main functional modules. All tests are expected to return `optimal`, confirming that the reduced network preserves the requested carbohydrate, lactate/pyruvate, and oxidative CO2-producing capabilities.